# PINN vs SSF: 60-Second Demo — No training required. Load pre-computed results and see the failures.

In [ ]:
# Install in Colab
!git clone https://github.com/NodeBox-Labs/pinn-vs-ssf-schrodinger.git
%cd pinn-vs-ssf-schrodinger
!pip install -q -r requirements.txt

In [ ]:
from src.reference.split_step import reference_solution
import yaml

with open("configs/case_d3_two_soliton_collision.yaml") as f:
    cfg = yaml.safe_load(f)
ref = reference_solution(cfg)
print(f"Reference computed on {len(ref['x'])} points, {len(ref['t'])} snapshots.")

In [ ]:
import torch
# Download from the v1.0.0 release artifacts
!wget -q https://github.com/NodeBox-Labs/pinn-vs-ssf-schrodinger/releases/download/v1.0.0/d3_two_soliton_collision_seed42.pt
ckpt = torch.load("d3_two_soliton_collision_seed42.pt", map_location="cpu")
print("PINN checkpoint loaded. Case:", ckpt["metrics"]["case"])
print("Rel-L2 final:", ckpt["metrics"]["rel_l2_final"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t_idx = -1  # final time
x = ref["x"]
psi_ref = ref["psi"][t_idx]
psi_pinn = ckpt["preds"][t_idx]

plt.figure(figsize=(10, 4))
plt.plot(x, np.abs(psi_ref)**2, lw=2, label="SSF (reference)")
plt.plot(x, np.abs(psi_pinn)**2, "--", lw=2, label=f"PINN (rel-L2 = {ckpt['metrics']['rel_l2_final']:.2f})")
plt.xlabel("x"); plt.ylabel("|ψ|²")
plt.title("Two-soliton collision at T=8 — the PINN fails to resolve the peaks")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()